# Train Root-Cause Classifier (ML, optional layer)

Loads `synthetic_failed_payments.csv` (from `data_generation.ipynb`), encodes features,
trains a LogisticRegression baseline, evaluates against the rule-based labels, and saves
the model bundle for `app/classifier.py::classify_failure_ml` to load.

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv('synthetic_failed_payments.csv')
print(f'Loaded {len(df)} rows')
df.head()


Loaded 325 rows


,payment_id,failure_code,failure_description,method,amount,true_scenario,root_cause,bank
0,pay_SIM247bd9e6df6a45,GATEWAY_ERROR,The card issuing bank server timed out.,card,299900,timeout,timeout,ICICI
1,pay_SIM9fa2fe2073024d,GATEWAY_ERROR,The card issuing bank server timed out.,netbanking,9900,timeout,timeout,AXIS
2,pay_SIMe2b5f0f87d8946,GATEWAY_ERROR,The card issuing bank server timed out.,upi,19900,timeout,timeout,ICICI
3,pay_SIM2dd946de48444a,GATEWAY_ERROR,The card issuing bank server timed out.,card,19900,timeout,timeout,HDFC
4,pay_SIM036bc2f74c0d40,GATEWAY_ERROR,The card issuing bank server timed out.,card,299900,timeout,timeout,SBI


## 1. Feature engineering

Match the feature set `classify_failure_ml()` in `app/classifier.py` builds at inference
time: `failure_code`, `method`, `amount_bucket`. We deliberately do NOT feed in
`failure_description` raw text — that's what the rule-based classifier already nails
deterministically; the point of the ML layer here is to see how well we can do WITHOUT
relying on the description string (useful if a payment gateway ever gives you a code but
a blank/unfamiliar description).

In [2]:
def amount_bucket(amount):
    if amount < 10000:
        return 'low'
    if amount < 100000:
        return 'mid'
    return 'high'

df['amount_bucket'] = df['amount'].apply(amount_bucket)

feature_cols = ['failure_code', 'method', 'amount_bucket']
target_col = 'root_cause'

encoders = {}
encoded = pd.DataFrame()
for col in feature_cols:
    le = LabelEncoder()
    encoded[col] = le.fit_transform(df[col])
    encoders[col] = le

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(df[target_col])
encoders['root_cause'] = target_encoder

X = encoded[feature_cols]
X.head()


,failure_code,method,amount_bucket
0,1,0,0
1,1,1,1
2,1,2,2
3,1,0,2
4,1,0,0


## 2. Train/test split + train baseline model

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'ML model accuracy on held-out set: {acc:.2%}')


ML model accuracy on held-out set: 48.78%


## 3. Evaluate: confusion matrix + per-class report

In [4]:
labels_present = sorted(set(y_test) | set(y_pred))
label_names = target_encoder.inverse_transform(labels_present)

print(classification_report(y_test, y_pred, labels=labels_present, target_names=label_names, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=labels_present)
pd.DataFrame(cm, index=label_names, columns=label_names)


                    precision    recall  f1-score   support

      auth_failure       0.62      0.38      0.48        13
      expired_card       0.00      0.00      0.00        10
insufficient_funds       0.47      0.75      0.58        20
      invalid_card       0.44      0.39      0.41        18
        risk_block       0.00      0.00      0.00         6
           timeout       0.50      0.87      0.63        15

          accuracy                           0.49        82
         macro avg       0.34      0.40      0.35        82
      weighted avg       0.40      0.49      0.42        82



,auth_failure,expired_card,insufficient_funds,invalid_card,risk_block,timeout
auth_failure,5,0,0,0,0,8
expired_card,0,0,6,4,0,0
insufficient_funds,0,0,15,5,0,0
invalid_card,0,0,11,7,0,0
risk_block,1,0,0,0,0,5
timeout,2,0,0,0,0,13


## 4. Compare against rule-based "accuracy"

Since our labels ARE the rule-based classifier's own output (see `data_generation.ipynb`),
the rule-based classifier is trivially 100% against this dataset — that's expected and not
a meaningful comparison on its own. The useful comparison is: **does the ML model, using
only `failure_code` + `method` + `amount_bucket` (no description text), recover the same
labels the rule-based classifier derives FROM the description text?** A high ML accuracy
here means failure_code alone is nearly as informative as the full description — useful to
know if you ever integrate a gateway that omits descriptions.

In [5]:
print(f'Rule-based classifier accuracy on its own labels: 100.00% (labels ARE its output)')
print(f'ML classifier accuracy (code+method+amount only): {acc:.2%}')
print()
print('Interpretation: the gap between these two numbers is how much signal is')
print('carried by failure_description text vs. failure_code/method/amount alone.')


Rule-based classifier accuracy on its own labels: 100.00% (labels ARE its output)
ML classifier accuracy (code+method+amount only): 48.78%

Interpretation: the gap between these two numbers is how much signal is
carried by failure_description text vs. failure_code/method/amount alone.


## 5. Save model bundle for app/classifier.py

In [6]:
bundle = {'model': model, 'encoders': encoders}
joblib.dump(bundle, 'root_cause_model.joblib')
print('Saved notebooks/root_cause_model.joblib')
print('app/classifier.py will auto-load this via CLASSIFIER_MODEL_PATH')


Saved notebooks/root_cause_model.joblib
app/classifier.py will auto-load this via CLASSIFIER_MODEL_PATH
